## Practice 2 · Manually Implementing Binary Cross-Entropy Loss and Cost Function
In class · Goal: run your first code on HaiMa and learn how to interrupt it.

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

In a binary classification neural network, the model usually gets the predicted probability that a sample belongs to the positive class through the Sigmoid function:
$$
\hat{y}=\sigma(w^Tx+b)
$$
Among them:

- \($\hat{y}$\) Shows the probability that the model predicts a positive class, ranging from 0 to 1;
- \(y\) Represents the actual label, with values of 0 or 1.

Binary cross-entropy loss is defined as:
$$
\mathcal{L}(\hat{y}, y) = -y\log(\hat{y}) -(1-y)\log(1-\hat{y})
$$

For all the training samples, the cost function is the average of the losses of all the samples:
$$
J(w,b) = \frac{1}{m}\sum_{i=1}^{m} \mathcal{L}\big(\hat{y}^{(i)}, y^{(i)}\big)
$$

Where (m) is the number of training samples.



## Question 1 · Manually Implement Binary Cross-Entropy Loss

Binary cross-entropy loss is defined as:

$$
\mathcal{L}(\hat{y}, y) = -y\log(\hat{y}) -(1-y)\log(1-\hat{y})
$$

where $\hat{y}$ is the predicted probability of the positive class (ranging from 0 to 1), and $y$ is the true label (0 or 1).

Please implement the function `bce_loss(y_hat, y)` in the answer cell below, which returns the binary cross-entropy loss for a single sample.

Use `math.log` to compute the logarithm, and add a small value `eps = 1e-10` to prevent `log(0)` errors.

**Hint**: When $y = 1$, the loss reduces to $-\log(\hat{y})$; when $y = 0$, the loss reduces to $-\log(1 - \hat{y})$.



In [1]:

import math

def bce_loss(y_hat, y):
    """
    Calculate the binary cross-entropy loss for a single sample.

    Parameters:
    y_hat: predicted probability of the positive class, in (0, 1)
    y: true label, 0 or 1

    Returns:
    Binary cross-entropy loss for a single sample
    """
    # 🟢 fill the loss function. You can use math.log to represent the log function.
    eps = 1e-10
    # YOUR CODE HERE
    loss =-y*math.log(y_hat+eps) - (1-y)*math.log(1-y_hat+eps)
    return loss

In [ ]:

# Cell type: Autograded tests. Points are set in this cell.


## Question 2 · An Intuitive Understanding of Binary Cross-Entropy Loss

Please just run this piece of code and directly observe how binary cross-entropy loss changes with predicted probability through text tables and graphs.


In [2]:
# ⚪ SKIP :just run 
#import matplotlib.pyplot
"""
Binary Cross-Entropy (BCE) Intuitive Demonstration
    L(y_hat, y) = -y*log(y_hat) - (1-y)*log(1-y_hat)
Core Intuition:
  * When y = 1, only -log(y_hat) takes effect.
    -> Loss approaches 0 when prediction is close to 1; loss rises sharply when prediction approaches 0.
  * When y = 0, only -log(1-y_hat) takes effect.
    -> Loss approaches 0 when prediction is close to 0; loss rises sharply when prediction approaches 1.
Therefore, BCE selects the corresponding logarithmic term according to the true label and penalizes its negative log-likelihood.
Run directly: python bce_demo.py
"""
import math
import sys
# Avoid Chinese garbled output in Windows console
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
EPS = 1e-12  # Prevent log(0) error (called eps / clip in real frameworks)
def bce(y_hat: float, y: int) -> float:
    """Binary cross-entropy loss. y_hat is model predicted probability of positive class, y is ground truth label (0 or 1)."""
    y_hat = min(max(y_hat, EPS), 1.0 - EPS)          # Value clipping for numerical stability
    return -(y * math.log(y_hat) + (1 - y) * math.log(1 - y_hat))
def bar(value: float, width: int = 46) -> str:
    """Visualize loss as horizontal bar for easy comparison (sqrt to compress scale, otherwise tiny values are invisible)."""
    n = int(round(math.sqrt(min(value / 40.0, 1.0)) * width))
    return "#" * n
# ============================================================================
# Demo 1: Two nested loops — outer loop iterates ground truth y, inner loop iterates predicted probability y_hat
# ============================================================================
y_hat_list = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 0.999]
print("=" * 78)
print("Binary Cross-Entropy Loss L(y_hat, y) = -y*log(y_hat) - (1-y)*log(1-y_hat)")
print("=" * 78)
for y in [1, 0]:                                   # Outer loop: ground truth label
    print(f"\n【Ground truth label y = {y}】The correct answer for this sample is {'positive class' if y == 1 else 'negative class'},"
          f" active term is -{'log(y_hat)' if y == 1 else 'log(1-y_hat)'}")
    print(f"{'y_hat':>7} | {'Loss':>8} | Loss magnitude")
    print("-" * 78)
    for y_hat in y_hat_list:                       # Inner loop: predicted probability
        loss = bce(y_hat, y)
        print(f"{y_hat:>7.3f} | {loss:>8.4f} | {bar(loss)}")
# ============================================================================
# Demo 2: Two nested loops — training perspective, see cost of wrong prediction and gradient direction
# ============================================================================
print("\n" + "=" * 78)
print("Training view: Loss comparison for same predictions under two labels (gap = severity of wrong prediction)")
print("=" * 78)
print(f"{'y_hat':>7} | {'L(y=1)':>9} | {'L(y=0)':>9} | {'|Diff|':>9}")
print("-" * 78)
for y_hat in y_hat_list:                           # Outer loop: predicted value
    row = []
    for y in [1, 0]:                               # Inner loop: two possible labels
        row.append(bce(y_hat, y))
    print(f"{y_hat:>7.3f} | {row[0]:>9.4f} | {row[1]:>9.4f} | {abs(row[0] - row[1]):>9.4f}")
# ============================================================================
# Demo 3: Batch average loss (in real training, loss is averaged over a mini-batch)
# ============================================================================
y_true = [1, 1, 0, 0, 1]
y_pred = [0.9, 0.2, 0.1, 0.8, 0.6]
print("\n" + "=" * 78)
print("Average loss over one mini-batch")
print("=" * 78)
print(f"{'Sample':>4} | {'y':>3} | {'y_hat':>7} | {'Loss':>8}")
print("-" * 78)
total = 0.0
for i, (t, p) in enumerate(zip(y_true, y_pred), start=1):
    loss = bce(p, t)
    total += loss
    print(f"{i:>4} | {t:>3} | {p:>7.3f} | {loss:>8.4f}")
print("-" * 78)
print(f"Average Loss = {total / len(y_true):.4f}   "
      f"(Note sample 2: ground truth y=1 but prediction y_hat=0.2, it has the largest loss)")
# ============================================================================
# Demo 4: Plot figure (automatically skip if matplotlib is not installed)
# ============================================================================
try:
    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xs = np.linspace(0.001, 0.999, 500)
    loss_y1 = -np.log(xs)
    loss_y0 = -np.log(1 - xs)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax1.plot(xs, loss_y1, lw=2.2, color="#d62728", label="y = 1:  -log(y_hat)")
    ax1.plot(xs, loss_y0, lw=2.2, color="#1f77b4", label="y = 0:  -log(1-y_hat)")
    ax1.set_ylim(0, 5)
    ax1.set_xlabel("y_hat  (predicted probability of class 1)")
    ax1.set_ylabel("loss")
    ax1.set_title("BCE vs. prediction")
    ax1.grid(alpha=0.3)
    ax1.legend()
    idx = [0, 2, 4, 6, 8, 10]
    labels = [f"{y_hat_list[i]:g}" for i in idx]
    w = 0.38
    pos = np.arange(len(idx))
    ax2.bar(pos - w / 2, [bce(y_hat_list[i], 1) for i in idx], w,
            label="y = 1", color="#d62728")
    ax2.bar(pos + w / 2, [bce(y_hat_list[i], 0) for i in idx], w,
            label="y = 0", color="#1f77b4")
    ax2.set_xticks(pos)
    ax2.set_xticklabels(labels)
    ax2.set_xlabel("y_hat")
    ax2.set_ylabel("loss")
    ax2.set_title("Same prediction, two labels")
    ax2.grid(alpha=0.3, axis="y")
    ax2.legend()
    fig.tight_layout()
    fig.savefig("bce_demo.png", dpi=140)
    print("\n[Figure] Saved to bce_demo.png")
except ImportError:
    print("\n[Hint] matplotlib/numpy not installed, plotting skipped. Text tables still work.")


Binary Cross-Entropy Loss L(y_hat, y) = -y*log(y_hat) - (1-y)*log(1-y_hat)

【Ground truth label y = 1】The correct answer for this sample is positive class, active term is -log(y_hat)
  y_hat |     Loss | Loss magnitude
------------------------------------------------------------------------------
  0.001 |   6.9078 | ###################
  0.010 |   4.6052 | ################
  0.050 |   2.9957 | #############
  0.100 |   2.3026 | ###########
  0.300 |   1.2040 | ########
  0.500 |   0.6931 | ######
  0.700 |   0.3567 | ####
  0.900 |   0.1054 | ##
  0.950 |   0.0513 | ##
  0.990 |   0.0101 | #
  0.999 |   0.0010 | 

【Ground truth label y = 0】The correct answer for this sample is negative class, active term is -log(1-y_hat)
  y_hat |     Loss | Loss magnitude
------------------------------------------------------------------------------
  0.001 |   0.0010 | 
  0.010 |   0.0101 | #
  0.050 |   0.0513 | ##
  0.100 |   0.1054 | ##
  0.300 |   0.3567 | ####
  0.500 |   0.6931 | ######
  0.70


## Question 3 · Implement BCE Loss and Cost Function

Binary Cross-Entropy (BCE) Loss is commonly used in binary classification tasks, defined as:

$$
\mathcal{L}(\hat{y}, y) = -y\log(\hat{y}) - (1-y)\log(1-\hat{y})
$$

where:
- $\hat{y}$ is the predicted probability of the positive class, ranging from 0 to 1;
- $y$ is the true label, taking values 0 or 1.

For all training samples, the cost function is the average of the losses:

$$
J(w,b) = \frac{1}{m}\sum_{i=1}^{m} \mathcal{L}\big(\hat{y}^{(i)}, y^{(i)}\big)
$$

Please complete **two** functions in the answer cell below:

### Task 1: `bce_loss(y_hat, y)`

Return the binary cross-entropy loss for a **single sample**.

Requirements:
1. Use **`np.log`** to compute the logarithm, so that both scalar and NumPy array inputs are supported.
2. Clip `y_hat` into the interval `[eps, 1 - eps]` with `eps = 1e-10` to prevent `log(0)` errors.
3. The function must correctly handle **both scalar inputs and NumPy array inputs (element-wise)**.

### Task 2: `cost_function(y_hat_list, y_list)`

Return the **average BCE loss** over all samples.

Requirements:
1. Use the `bce_loss` function implemented above.
2. Use **NumPy** to convert the input lists into arrays and compute the mean loss.
3. Return a single `float` value.

**Hints**:
- When $y = 1$, the loss reduces to $-\log(\hat{y})$; when $y = 0$, the loss reduces to $-\log(1 - \hat{y})$.
- Use `np.mean` for the average.


In [ ]:

import math
import numpy as np

def bce_loss(y_hat, y):
    """
    Calculate the binary cross-entropy loss for a single sample.
    Supports both scalar and NumPy array inputs.

    Parameters:
    y_hat: Predicted probability of the positive class, in [0, 1]
    y: True label, 0 or 1

    Returns:
    Binary cross-entropy loss for a single sample
    """
    eps = 1e-10
    # 🟢 EDIT | change this line
    # YOUR CODE HERE
    y_hat = np.clip(np.asarray(y_hat, dtype=float), eps, 1 - eps)
    y = np.asarray(y, dtype=float)

    loss = -y * np.log(y_hat) - (1 - y) * np.log(1 - y_hat)
    return loss


def cost_function(y_hat_list, y_list):
    """
    Calculate the average BCE loss over all samples.

    Parameters:
    y_hat_list: list of predicted probabilities
    y_list: list of true labels

    Returns:
    Average BCE loss (float)
    """
    # 🟢 EDIT | change this line
    # YOUR CODE HERE
    y_hat_list = np.asarray(y_hat_list, dtype=float)
    y_list = np.asarray(y_list, dtype=float)

    return float(np.mean(bce_loss(y_hat_list, y_list)))



In [ ]:

# Cell type: Autograded tests. Points are set in this cell.
